# Evaluación comparativa: semantic vs lexical AND→OR vs hybrid

Este notebook compara **semantic search**, **lexical search en dos etapas** y **hybrid search** usando el mismo evidence set final. La rama híbrida fusiona mediante Reciprocal Rank Fusion (RRF) exactamente las dos ramas independientes evaluadas por separado.

## Seguridad y alcance

- Este notebook es una copia nueva y no modifica código fuente original, documentos, CSV gold ni la base de datos.
- `RUN_RETRIEVAL` empieza en `False`: no se consulta PostgreSQL ni se generan embeddings hasta activarlo explícitamente.
- Los retrievers de v5 usan exclusivamente consultas `SELECT`; no se llaman funciones de ingesta ni escritura.
- No se imprimen variables de entorno, cadenas de conexión ni secretos.
- Al ejecutar las celdas de guardado se crean CSV nuevos con el sufijo `21082026_2027`; no se sobrescriben los resultados de las 19:30.

## Dos vistas del gold

1. **human_only**: solo evidencias con `review_status = human_reviewed`. Es la vista más rigurosa.
2. **expanded**: incorpora también `machine_only_unreviewed`. Aporta cobertura, pero sus resultados son análisis de sensibilidad, no gold humano.

La unidad principal es el **chunk/evidence**. Se incluyen Document y Page Hit@k como análisis secundarios.

## Protocolo de ejecución

1. Ejecuta las celdas de configuración y carga del gold.
2. Revisa que las rutas y parámetros sean correctos.
3. Cambia `RUN_RETRIEVAL` a `True` solo si autorizas las consultas de lectura y el uso del modelo de embeddings ya configurado.
4. Recupera un pool de 100 candidatos por pregunta con semantic, lexical e hybrid.
5. En lexical, ejecuta primero una consulta AND estricta; si no completa el pool, añade candidatos de un fallback OR.
6. Conserva y revisa por separado cuántos candidatos proceden de AND y cuántos de OR.
7. Calcula las métricas de salida a `k = 1, 3, 5, 10`.
8. Interpreta por separado las vistas `human_only` y `expanded`, tanto por pregunta como de forma agregada.
9. Usa las métricas por pregunta para cualquier prueba estadística pareada.

El notebook conserva la etapa léxica de procedencia de cada candidato. Esto permite comprobar si el OR aumenta recall a costa de introducir ruido o empeorar los primeros rangos.

# Cambios principales respecto a versiones anteriores

Esta versión usa `app.retrieval_revised_v5`. Mantiene la detección local de idioma con **Lingua** incorporada en v4 y añade una recuperación léxica en dos etapas:

1. **AND estricto** mediante `websearch_to_tsquery`.
2. **OR como fallback** cuando AND devuelve menos candidatos que la profundidad solicitada.

Los candidatos AND mantienen siempre prioridad. El fallback OR solo completa las posiciones libres, elimina duplicados por `(doc_id, chunk_id)` y registra en cada resultado si procede de `and` o de `or_fallback`.

Este cambio se introduce porque la consulta AND de v4 devolvía cero candidatos en muchas preguntas. La nueva instrumentación permite comprobar si relajar la consulta mejora la cobertura sin confundir esa mejora con la detección de idioma o con la fusión híbrida.

In [1]:
# Importaciones. No realizan recuperación, acceso a base de datos ni escritura.
from __future__ import annotations

from dataclasses import asdict
from itertools import combinations
from pathlib import Path
import importlib.metadata
import platform
import sys

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

try:
    from scipy.stats import friedmanchisquare, wilcoxon
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    print('SciPy no está disponible: las métricas funcionarán, pero no los tests estadísticos.')


In [ ]:
# Configuración reproducible. Cambia valores solo antes de ejecutar retrieval.
def find_repo_root(start: Path) -> Path:
    """Busca el directorio del repositorio sin asumir desde qué carpeta se abrió Jupyter."""
    for candidate in (start, *start.parents):
        if (candidate / 'app').is_dir() and (candidate / 'evaluation_v2').is_dir():
            return candidate
    raise FileNotFoundError('No se localizó la raíz del repositorio RAGChatBot.')

ROOT = find_repo_root(Path.cwd().resolve())
FINAL_DIR = ROOT / 'evaluation_v2' / 'run_20260809_170051' / 'final_single_reviewer_v2'
QUESTIONS_PATH = ROOT / 'evaluation_v2' / 'run_20260809_170051' / 'questions.csv'
FINAL_WORKBOOK_PATH = FINAL_DIR / 'evaluation_final_single_reviewer.xlsx'

EVIDENCE_SHEET = 'Evidence Final'
CLAIMS_SHEET = 'Claims Final'
LINKS_SHEET = 'Evidence Claim Links'

# Mantener False evita consultas SELECT y llamadas al modelo de embeddings.
RUN_RETRIEVAL = False
# Los mismos cortes se aplican a las tres estrategias.
CUTOFFS = [1, 3, 5, 10]
MAX_K = max(CUTOFFS)
RETRIEVAL_DEPTH = 100  # Pool común; las métricas se calculan solo en CUTOFFS.
RRF_K = 60
TOPIC_FILTER = None  # Usa None para no aplicar un filtro que sesgue la comparación.
STRATEGIES = ['semantic', 'lexical', 'hybrid']

# Cobertura de claims: 'required_only' usa solo claims necesarios para responder completamente.
CLAIM_SCOPE = 'all_active'  # Alternativa: 'all_active' o 'required_only'

# Métrica primaria predefinida para el test estadístico.
STAT_GOLD_VIEW = 'expanded'
STAT_METRIC = 'claim_coverage_at_k'
STAT_K = 5
ALPHA = 0.05

for required_path in [QUESTIONS_PATH, FINAL_WORKBOOK_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'Falta un archivo requerido: {required_path}')

print(f'Raíz: {ROOT}')
print(f'Evidence set: {FINAL_WORKBOOK_PATH.name} / {EVIDENCE_SHEET}')
print(f'RUN_RETRIEVAL = {RUN_RETRIEVAL}')


Raíz: C:\Users\mamen\Documents\Python\RAGChatBot
Evidence set: evaluation_final_single_reviewer.xlsx / Evidence Final
RUN_RETRIEVAL = True


In [3]:
# Se usa la copia revisada para comparar una ablación limpia: semantic, lexical e hybrid.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app.retrieval_revised_v5 import search, lexical_search, hybrid_search, resolve_lexical_language

c:\Users\mamen\anaconda3\envs\RAGChatBot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Se leen los archivos finales sin modificarlos. dtype='string' protege IDs y valores vacíos.
questions = pd.read_csv(QUESTIONS_PATH, dtype='string', keep_default_na=False)

evidence = pd.read_excel(
    FINAL_WORKBOOK_PATH,
    sheet_name=EVIDENCE_SHEET,
    dtype='string',
    keep_default_na=False,
)

claims = pd.read_excel(
    FINAL_WORKBOOK_PATH,
    sheet_name=CLAIMS_SHEET,
    dtype='string',
    keep_default_na=False,
)

links = pd.read_excel(
    FINAL_WORKBOOK_PATH,
    sheet_name=LINKS_SHEET,
    dtype='string',
    keep_default_na=False,
)

evidence['relevance_grade'] = pd.to_numeric(evidence['relevance_grade'], errors='raise')
claims['required_for_complete_answer'] = claims['required_for_complete_answer'].str.lower().eq('true')

print(f'Preguntas: {len(questions)}')
print(f'Evidencias: {len(evidence)}')
print(f'Claims activos: {len(claims)}')
display(evidence.groupby('review_status', dropna=False).size().rename('n').reset_index())


Preguntas: 31
Evidencias: 162
Claims activos: 152


,review_status,n
0,human_reviewed,40
1,machine_only_unreviewed,122


## Métricas de retrieval

Sea $r_i=1$ si el resultado en rango $i$ es una evidencia relevante y $0$ si no. En este notebook, para las métricas binarias una evidencia es relevante cuando su grado es $\geq 2$.

### Evidence Hit@k

$$Hit@k(q)=\mathbb{1}[\exists i\leq k: r_i=1]$$

Indica si la pregunta tiene al menos una evidencia útil entre los primeros $k$ resultados.

### Evidence Precision@k

$$P@k(q)=\frac{\sum_{i=1}^{k}r_i}{k}$$

Mide qué parte del contexto recuperado es útil. El denominador es siempre $k$: si se devuelven menos resultados, los rangos ausentes se tratan como no relevantes.

### Evidence Recall@k

$$Recall@k(q)=\frac{|G_q \cap R_q^{@k}|}{|G_q|}$$

$G_q$ es el conjunto de evidencias gold para la pregunta y $R_q^{@k}$ las recuperadas en top-$k$. Solo se agregan preguntas que tengan al menos una evidencia evaluable en la vista gold elegida.

### MRR

$$RR(q)=\frac{1}{\min\{i:r_i=1\}},\qquad MRR=\frac{1}{|Q|}\sum_{q\in Q}RR(q)$$

Premia encontrar la primera evidencia relevante en los primeros puestos.

### AP y MAP

$$AP@k(q)=\frac{1}{|G_q|}\sum_{i=1}^{k}P@i(q)r_i,\qquad MAP@k=\frac{1}{|Q|}\sum_{q\in Q}AP@k(q)$$

AP valora todas las evidencias relevantes y las posiciones en que aparecen; MAP es su media entre preguntas.

### Claim Coverage@k

$$ClaimCoverage@k(q)=\frac{|C_q\cap C(R_q^{@k})|}{|C_q|}$$

$C_q$ son los claims evaluables de la pregunta y $C(R_q^{@k})$ son los claims enlazados a evidencias recuperadas. Esta es la métrica más cercana a la utilidad del contexto RAG.

### Document y Page Hit@k

Usan la misma fórmula que Hit@k, pero consideran correcto respectivamente un `doc_id` gold o el par `(doc_id, page_num)` gold. Son complementarias: el análisis principal sigue siendo evidence/chunk-level.

In [5]:
# Funciones para construir las dos vistas del gold y las claves de relevancia.
def split_ids(value: str) -> set[str]:
    """Convierte listas separadas por ; o , en un conjunto de IDs sin duplicados."""
    return {part.strip() for part in str(value).replace(',', ';').split(';') if part.strip()}

def make_evidence_key(row: pd.Series) -> tuple:
    """Usa (doc_id, chunk_id); chunk_id por sí solo no tiene por qué ser globalmente único."""
    doc_id = str(row.get('doc_id', '')).strip()
    chunk_id = str(row.get('chunk_id', '')).strip()
    if chunk_id:
        return ('doc_chunk', doc_id, chunk_id)
    return ('doc_page', doc_id, str(row.get('page_num', '')).strip())

def make_gold_view(view_name: str) -> dict:
    """Prepara índices gold de evidencia, documento, página y claim para una vista."""
    if view_name == 'human_only':
        selected_evidence = evidence.loc[evidence['review_status'].eq('human_reviewed')].copy()
    elif view_name == 'expanded':
        selected_evidence = evidence.copy()
    else:
        raise ValueError("view_name debe ser 'human_only' o 'expanded'")

    selected_evidence = selected_evidence.loc[selected_evidence['relevance_grade'].ge(2)].copy()
    selected_evidence['evidence_key'] = selected_evidence.apply(make_evidence_key, axis=1)
    selected_ids = set(selected_evidence['evidence_id'])
    selected_links = links.loc[links['evidence_id'].isin(selected_ids)].copy()

    scoped_claims = claims.copy()
    if CLAIM_SCOPE == 'required_only':
        scoped_claims = scoped_claims.loc[scoped_claims['required_for_complete_answer']].copy()
    eligible_claim_ids = set(selected_links['claim_id']) & set(scoped_claims['claim_id'])
    selected_links = selected_links.loc[selected_links['claim_id'].isin(eligible_claim_ids)].copy()

    by_question = {}
    for question_id, group in selected_evidence.groupby('question_id', sort=False):
        group_links = selected_links.loc[selected_links['evidence_id'].isin(set(group['evidence_id']))]
        evidence_to_claims = group_links.groupby('evidence_id')['claim_id'].agg(lambda x: set(x)).to_dict()
        key_to_evidence_ids = group.groupby('evidence_key')['evidence_id'].agg(lambda x: set(x)).to_dict()
        evidence_to_grade = group.groupby('evidence_id')['relevance_grade'].max().to_dict()
        by_question[question_id] = {
            'gold_keys': set(group['evidence_key']),
            'gold_doc_ids': set(group['doc_id'].astype(str)),
            'gold_pages': set(zip(group['doc_id'].astype(str), group['page_num'].astype(str))),
            'key_to_evidence_ids': key_to_evidence_ids,
            'evidence_to_claims': evidence_to_claims,
            'evidence_to_grade': evidence_to_grade,
            'eligible_claim_ids': set(group_links['claim_id']),
        }
    return {'name': view_name, 'evidence': selected_evidence, 'links': selected_links, 'by_question': by_question}

gold_views = {name: make_gold_view(name) for name in ['human_only', 'expanded']}
for name, view in gold_views.items():
    evaluable_questions = sum(bool(item['gold_keys']) for item in view['by_question'].values())
    print(f'{name}: {len(view["evidence"])} evidencias, {evaluable_questions} preguntas con evidencia gold')


human_only: 40 evidencias, 18 preguntas con evidencia gold
expanded: 162 evidencias, 31 preguntas con evidencia gold


## Recuperación controlada

Las tres estrategias reciben exactamente la misma pregunta, profundidad y filtro temático:

- `semantic` usa solamente pgvector.
- `lexical` usa PostgreSQL Full-Text Search en dos etapas.
- `hybrid` fusiona mediante RRF exactamente los candidatos de las dos ramas anteriores.

### Dos etapas de la búsqueda léxica

La primera etapa utiliza `websearch_to_tsquery` y actúa como una consulta AND estricta. Si devuelve menos candidatos que `RETRIEVAL_DEPTH`, v5 construye una segunda consulta con los lexemas normalizados por PostgreSQL unidos mediante OR.

Los resultados AND se colocan antes que los OR. Los chunks repetidos se eliminan por `(doc_id, chunk_id)` y cada candidato conserva:

- `lexical_query_mode`: `and` u `or_fallback`;
- `lexical_and_candidates`: candidatos obtenidos por la etapa estricta;
- `lexical_or_candidates_added`: candidatos nuevos añadidos por OR;
- `lexical_or_fallback_used`: si fue necesario ejecutar la etapa relajada.

La detección de idioma se realiza una sola vez con Lingua y la misma resolución lingüística se reutiliza en lexical e hybrid. Si el idioma no está soportado o no puede determinarse, se registra el fallback a la configuración PostgreSQL `simple`.

La celda siguiente solo ejecuta retrieval cuando `RUN_RETRIEVAL = True`. Todas las consultas sobre `rag_chunks` son de lectura.

In [6]:
# Probar la detección de idioma y la resolución de configuración FTS léxica.

examples = [
    "¿Cuáles son los efectos secundarios de la quimioterapia?",
    "¿Qué es la neutropenia?",
    "¿Para qué sirve el Radium-223?",
    "What is fatigue?",
    "これは何ですか？",
    "",
]

for query in examples:
    print(query, resolve_lexical_language(query))

¿Cuáles son los efectos secundarios de la quimioterapia? LexicalLanguageResolution(detected_language='es', postgres_config='spanish', supported=True, fallback_to_simple=False, note="Using PostgreSQL 'spanish' configuration.", detector_name='lingua', confidence=0.9721277124087917, confidence_margin=0.94630375138539)
¿Qué es la neutropenia? LexicalLanguageResolution(detected_language='pt', postgres_config='simple', supported=False, fallback_to_simple=True, note="Language 'pt' is not supported by the retrieval configuration; defaulting to simple.", detector_name='lingua', confidence=0.37378127556534413, confidence_margin=0.1262547572146749)
¿Para qué sirve el Radium-223? LexicalLanguageResolution(detected_language='es', postgres_config='spanish', supported=True, fallback_to_simple=False, note="Using PostgreSQL 'spanish' configuration.", detector_name='lingua', confidence=0.3461656514095611, confidence_margin=0.03684823522240582)
What is fatigue? LexicalLanguageResolution(detected_language

In [7]:
# Esta es la única celda que llama a los tres retrievers. Todas las ramas son de solo lectura.
if RUN_RETRIEVAL:

    RETRIEVERS = {
        "semantic": lambda query, language_resolution: search(
            query,
            top_k=RETRIEVAL_DEPTH,
            topic=TOPIC_FILTER,
        ),
        "lexical": lambda query, language_resolution: lexical_search(
            query,
            top_k=RETRIEVAL_DEPTH,
            topic=TOPIC_FILTER,
            language_resolution=language_resolution,
        ),
        "hybrid": lambda query, language_resolution: hybrid_search(
            query,
            top_k=RETRIEVAL_DEPTH,
            topic=TOPIC_FILTER,
            rrf_k=RRF_K,
            candidate_k=RETRIEVAL_DEPTH,
            language_resolution=language_resolution,
        ),
    }

    retrieval_rows = []
    language_diagnostic_rows = []

    for question_row in questions.itertuples(index=False):
        question_id = str(question_row.question_id)
        query = str(question_row.question)

        # Se resuelve una sola vez y se reutiliza en lexical e hybrid.
        language_resolution = resolve_lexical_language(query)

        language_diagnostic_rows.append({
            "question_id": question_id,
            "detector_name": language_resolution.detector_name,
            "detected_language": language_resolution.detected_language,
            "language_confidence": language_resolution.confidence,
            "language_confidence_margin": language_resolution.confidence_margin,
            "postgres_ts_config": language_resolution.postgres_config,
            "language_supported": language_resolution.supported,
            "fallback_to_simple": language_resolution.fallback_to_simple,
            "language_note": language_resolution.note})
        
        for strategy, retrieve in RETRIEVERS.items():
            # Cada llamada devuelve datos en memoria; no se insertan ni actualizan registros.
            retrieved_chunks = retrieve(query, language_resolution)
            seen = set()
            final_rank = 0
            for original_rank, chunk in enumerate(retrieved_chunks, start=1):
                item = asdict(chunk)
                dedupe_key = (str(item['doc_id']), str(item['chunk_id']))
                if dedupe_key in seen:
                    continue
                seen.add(dedupe_key)
                final_rank += 1
                retrieval_rows.append({
                    'question_id': question_id,
                    'question': query,
                    'strategy': strategy,
                    'rank': final_rank,
                    'original_rank': original_rank,
                    'doc_id': item['doc_id'],
                    'chunk_id': item['chunk_id'],
                    'page_num': item['page_num'],
                    'chunk_lang': item['lang'],
                    # Los campos siguientes permiten auditar qué señal produjo cada ranking.
                    'raw_distance': item.get('distance'),
                    'raw_score': item.get('score'),
                    'semantic_rank': item.get('semantic_rank'),
                    'lexical_rank': item.get('lexical_rank'),
                    'semantic_distance': item.get('semantic_distance'),
                    'lexical_score': item.get('lexical_score'),
                    'ts_config': item.get('ts_config'),
                    'rrf_k': RRF_K if strategy == 'hybrid' else np.nan,
                    'candidate_depth': RETRIEVAL_DEPTH,
                    "detector_name": language_resolution.detector_name,
                    "detected_language": language_resolution.detected_language,
                    "language_confidence": language_resolution.confidence,
                    "language_confidence_margin": language_resolution.confidence_margin,
                    "postgres_ts_config": language_resolution.postgres_config,
                    "language_supported": language_resolution.supported,
                    "fallback_to_simple": language_resolution.fallback_to_simple,
                    "language_note": language_resolution.note,
                    # Diagnóstico de la nueva recuperación léxica en dos etapas.
                    "lexical_query_mode": item.get("lexical_query_mode"),
                    "lexical_and_candidates": item.get("lexical_and_candidates"),
                    "lexical_or_candidates_added": item.get(
                        "lexical_or_candidates_added"
                    ),
                    "lexical_or_fallback_used": item.get(
                        "lexical_or_fallback_used"
                    )})

    retrieval_results = pd.DataFrame(retrieval_rows)
    if retrieval_results.empty:
        raise RuntimeError('Los retrievers no devolvieron resultados; revisa el servicio y la configuración.')
    display(retrieval_results.head())
else:
    print('Retrieval no ejecutado. Cambia RUN_RETRIEVAL a True para crear retrieval_results en memoria.')


,question_id,question,strategy,rank,original_rank,doc_id,chunk_id,page_num,chunk_lang,raw_distance,...,language_confidence,language_confidence_margin,postgres_ts_config,language_supported,fallback_to_simple,language_note,lexical_query_mode,lexical_and_candidates,lexical_or_candidates_added,lexical_or_fallback_used
0,Q001,¿Qué es la astenia?,semantic,1,1,general_es_gepac_guia-toxicidad-quimioterapia_v1,p009_c00,9,es,0.506965,...,0.464878,0.187137,simple,False,True,Language 'ca' is not supported by the retrieva...,NaN,NaN,NaN,None
1,Q001,¿Qué es la astenia?,semantic,2,2,prostata_es_gepac_guia-cancer-de-prostata_2020,p014_c00,14,es,0.527896,...,0.464878,0.187137,simple,False,True,Language 'ca' is not supported by the retrieva...,NaN,NaN,NaN,None
2,Q001,¿Qué es la astenia?,semantic,3,3,general_es_gepac_guia-toxicidad-quimioterapia_v1,p004_c00,4,es,0.609280,...,0.464878,0.187137,simple,False,True,Language 'ca' is not supported by the retrieva...,NaN,NaN,NaN,None
3,Q001,¿Qué es la astenia?,semantic,4,4,mama_es_hureinasofia_protocolo-cancer-mama_2021,p034_c00,34,es,0.613692,...,0.464878,0.187137,simple,False,True,Language 'ca' is not supported by the retrieva...,NaN,NaN,NaN,None
4,Q001,¿Qué es la astenia?,semantic,5,5,prostata_es_gepac_guia-cancer-de-prostata_2020,p177_c00,177,es,0.626868,...,0.464878,0.187137,simple,False,True,Language 'ca' is not supported by the retrieva...,NaN,NaN,NaN,None


In [8]:
retrieval_results.info()

<class 'pandas.DataFrame'>
RangeIndex: 9050 entries, 0 to 9049
Data columns (total 30 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   question_id                  9050 non-null   str    
 1   question                     9050 non-null   str    
 2   strategy                     9050 non-null   str    
 3   rank                         9050 non-null   int64  
 4   original_rank                9050 non-null   int64  
 5   doc_id                       9050 non-null   str    
 6   chunk_id                     9050 non-null   str    
 7   page_num                     9050 non-null   int64  
 8   chunk_lang                   9050 non-null   str    
 9   raw_distance                 3100 non-null   float64
 10  raw_score                    5950 non-null   float64
 11  semantic_rank                5044 non-null   float64
 12  lexical_rank                 4589 non-null   float64
 13  semantic_distance            

In [9]:
retrieval_results_path = FINAL_DIR / 'retrieval_results_21082026_2027.csv'
retrieval_results.to_csv(retrieval_results_path, index=False)
print(f'Resultados de retrieval guardados en: {retrieval_results_path}')

Resultados de retrieval guardados en: C:\Users\mamen\Documents\Python\RAGChatBot\evaluation_v2\run_20260809_170051\final_single_reviewer_v2\retrieval_results_21082026_2027.csv


In [10]:
# Diagnóstico por pregunta de la detección de idioma y de las dos etapas FTS.
lexical_language_diagnostics = pd.DataFrame(language_diagnostic_rows)

lexical_rows = retrieval_results.loc[
    retrieval_results["strategy"].eq("lexical")
].copy()

# Total final y candidatos aportados por cada etapa.
lexical_candidate_counts = lexical_rows.groupby("question_id").size()

and_candidate_counts = (
    lexical_rows.loc[
        lexical_rows["lexical_query_mode"].eq("and")
    ]
    .groupby("question_id")
    .size()
)

or_candidate_counts = (
    lexical_rows.loc[
        lexical_rows["lexical_query_mode"].eq("or_fallback")
    ]
    .groupby("question_id")
    .size()
)

hybrid_lexical_candidate_counts = (
    retrieval_results.loc[
        retrieval_results["strategy"].eq("hybrid")
        & retrieval_results["lexical_rank"].notna()
    ]
    .groupby("question_id")
    .size()
)

lexical_language_diagnostics["lexical_candidates_returned"] = (
    lexical_language_diagnostics["question_id"]
    .map(lexical_candidate_counts)
    .fillna(0)
    .astype(int)
)

lexical_language_diagnostics["lexical_and_candidates"] = (
    lexical_language_diagnostics["question_id"]
    .map(and_candidate_counts)
    .fillna(0)
    .astype(int)
)

lexical_language_diagnostics["lexical_or_candidates_added"] = (
    lexical_language_diagnostics["question_id"]
    .map(or_candidate_counts)
    .fillna(0)
    .astype(int)
)

# V5 ejecuta OR siempre que AND no llena el pool solicitado, incluso si OR
# tampoco encuentra candidatos nuevos.
lexical_language_diagnostics["lexical_or_fallback_used"] = (
    lexical_language_diagnostics["lexical_and_candidates"]
    .lt(RETRIEVAL_DEPTH)
)

lexical_language_diagnostics["hybrid_candidates_with_lexical_rank"] = (
    lexical_language_diagnostics["question_id"]
    .map(hybrid_lexical_candidate_counts)
    .fillna(0)
    .astype(int)
)

# Comprobación interna: el total final debe ser la suma de AND y OR deduplicados.
calculated_total = (
    lexical_language_diagnostics["lexical_and_candidates"]
    + lexical_language_diagnostics["lexical_or_candidates_added"]
)

if not calculated_total.equals(
    lexical_language_diagnostics["lexical_candidates_returned"]
):
    raise AssertionError(
        "Los candidatos AND y OR no suman el total lexical."
    )

display(lexical_language_diagnostics)

,question_id,detector_name,detected_language,language_confidence,language_confidence_margin,postgres_ts_config,language_supported,fallback_to_simple,language_note,lexical_candidates_returned,lexical_and_candidates,lexical_or_candidates_added,lexical_or_fallback_used,hybrid_candidates_with_lexical_rank
0,Q001,lingua,ca,0.464878,0.187137,simple,False,True,Language 'ca' is not supported by the retrieva...,100,0,100,True,57
1,Q002,lingua,es,0.855018,0.760259,spanish,True,False,Using PostgreSQL 'spanish' configuration.,100,0,100,True,55
2,Q003,lingua,es,0.996287,0.993915,spanish,True,False,Using PostgreSQL 'spanish' configuration.,100,0,100,True,59
3,Q004,lingua,es,0.502377,0.246597,spanish,True,False,Using PostgreSQL 'spanish' configuration.,100,0,100,True,55
4,Q005,lingua,es,0.997027,0.994338,spanish,True,False,Using PostgreSQL 'spanish' configuration.,100,0,100,True,59
5,Q006,lingua,pt,0.373781,0.126255,simple,False,True,Language 'pt' is not supported by the retrieva...,100,1,99,True,54
6,Q007,lingua,es,0.964809,0.933851,spanish,True,False,Using PostgreSQL 'spanish' configuration.,100,1,99,True,64
7,Q008,lingua,es,0.956934,0.922861,spanish,True,False,Using PostgreSQL 'spanish' configuration.,100,0,100,True,56
8,Q009,lingua,es,0.925259,0.856701,spanish,True,False,Using PostgreSQL 'spanish' configuration.,100,0,100,True,57
9,Q010,lingua,es,0.686146,0.465352,spanish,True,False,Using PostgreSQL 'spanish' configuration.,100,0,100,True,59


In [11]:
lexical_language_diagnostics_path = FINAL_DIR / 'lexical_language_diagnostics_21082026_2027.csv'
lexical_language_diagnostics.to_csv(lexical_language_diagnostics_path, index=False)
print(f'Diagnósticos de detección de idioma y resolución léxica guardados en: {lexical_language_diagnostics_path}')

Diagnósticos de detección de idioma y resolución léxica guardados en: C:\Users\mamen\Documents\Python\RAGChatBot\evaluation_v2\run_20260809_170051\final_single_reviewer_v2\lexical_language_diagnostics_21082026_2027.csv


In [12]:
#Fallback cases 
fallback_cases = lexical_language_diagnostics.loc[
    lexical_language_diagnostics["fallback_to_simple"]
]

if fallback_cases.empty:
    print("Todas las consultas utilizan una configuración PostgreSQL soportada.")
else:
    print(
        f"{len(fallback_cases)} consultas usan fallback a la configuración 'simple'."
    )
    display(
        fallback_cases[
            [
                "question_id",
                "detected_language",
                "postgres_ts_config",
                "language_note",
            ]
        ]
    )

4 consultas usan fallback a la configuración 'simple'.


,question_id,detected_language,postgres_ts_config,language_note
0,Q001,ca,simple,Language 'ca' is not supported by the retrieva...
5,Q006,pt,simple,Language 'pt' is not supported by the retrieva...
29,Q030,fr,simple,Language 'fr' is not supported by the retrieva...
30,Q031,ca,simple,Language 'ca' is not supported by the retrieva...


Lingua y el fallback FTS deben evaluarse como componentes diferentes. Las columnas de diagnóstico permiten separar:

- errores de detección del idioma;
- consultas AND que no encuentran candidatos;
- candidatos adicionales recuperados mediante OR;
- consultas donde OR tampoco completa el pool.

No debe atribuirse automáticamente a Lingua una mejora producida por el fallback OR, ni atribuir al OR un fallo causado por una configuración lingüística incorrecta.

In [13]:
# Utilidades de matching: identifican chunks por (doc_id, chunk_id) y usan doc_id+página como fallback.
def result_evidence_key(row: pd.Series) -> tuple:
    doc_id = str(row.get('doc_id', '')).strip()
    chunk_id = str(row.get('chunk_id', '')).strip()
    if chunk_id:
        return ('doc_chunk', doc_id, chunk_id)
    return ('doc_page', doc_id, str(row.get('page_num', '')).strip())

def matched_evidence_ids(result_row: pd.Series, gold_for_question: dict) -> set[str]:
    """Devuelve todas las evidencias gold equivalentes a un resultado recuperado."""
    result_key = result_evidence_key(result_row)
    return set(gold_for_question['key_to_evidence_ids'].get(result_key, set()))

def evaluate_one_query(ranking: pd.DataFrame, gold_for_question: dict, cutoff: int) -> dict:
    """Calcula todas las métricas para una pregunta, estrategia y cutoff."""
    top = ranking.sort_values('rank').head(cutoff).copy()
    gold_keys = gold_for_question['gold_keys']
    gold_docs = gold_for_question['gold_doc_ids']
    gold_pages = gold_for_question['gold_pages']
    eligible_claims = gold_for_question['eligible_claim_ids']

    retrieved_gold_keys = set()
    covered_claims = set()
    binary_relevance = []

    for _, row in top.iterrows():
        evidence_ids = matched_evidence_ids(row, gold_for_question)
        is_relevant = bool(evidence_ids)
        binary_relevance.append(int(is_relevant))
        if is_relevant:
            retrieved_gold_keys.add(result_evidence_key(row))
            for evidence_id in evidence_ids:
                covered_claims.update(gold_for_question['evidence_to_claims'].get(evidence_id, set()))

    # Las posiciones no devueltas se añaden como no relevantes para mantener P@k con denominador fijo.
    binary_relevance += [0] * max(0, cutoff - len(binary_relevance))
    rel = np.asarray(binary_relevance, dtype=int)
    ranks = np.arange(1, cutoff + 1)
    precision_prefix = np.cumsum(rel) / ranks
    first_relevant = np.flatnonzero(rel)

    num_gold = len(gold_keys)
    num_claims = len(eligible_claims)
    return {
        'evaluable_evidence': num_gold > 0,
        'num_gold_evidence': num_gold,
        'num_gold_claims': num_claims,
        'evidence_hit_at_k': float(rel.any()),
        'evidence_precision_at_k': float(rel.sum() / cutoff),
        'evidence_recall_at_k': float(len(retrieved_gold_keys) / num_gold) if num_gold else np.nan,
        'mrr_at_k': float(1 / (first_relevant[0] + 1)) if len(first_relevant) else 0.0,
        'ap_at_k': float((precision_prefix * rel).sum() / num_gold) if num_gold else np.nan,
        'claim_coverage_at_k': float(len(covered_claims & eligible_claims) / num_claims) if num_claims else np.nan,
        'document_hit_at_k': float(any(str(row.doc_id) in gold_docs for row in top.itertuples(index=False))),
        'page_hit_at_k': float(any((str(row.doc_id), str(row.page_num)) in gold_pages for row in top.itertuples(index=False))),
        'retrieved_relevant_evidence': int(rel.sum()),
        'retrieved_claims': ';'.join(sorted(covered_claims & eligible_claims)),
    }


In [14]:
# Calcula métricas por pregunta y solo después agrega. Requiere ejecutar la celda de retrieval.
if 'retrieval_results' not in globals():
    raise RuntimeError('No existe retrieval_results. Activa RUN_RETRIEVAL y ejecuta la celda de recuperación primero.')

metric_rows = []
for gold_view_name, gold_view in gold_views.items():
    for question_row in questions.itertuples(index=False):
        question_id = str(question_row.question_id)
        gold_for_question = gold_view['by_question'].get(question_id)
        if gold_for_question is None:
            # No se inventan negativos cuando el gold no tiene evidencia para esa pregunta.
            continue
        question_results = retrieval_results.loc[retrieval_results['question_id'].eq(question_id)]
        # Iterar sobre STRATEGIES conserva también una estrategia que devuelva cero resultados.
        # Así no se eliminan silenciosamente consultas difíciles del promedio ni de los tests pareados.
        for strategy in STRATEGIES:
            ranking = question_results.loc[question_results['strategy'].eq(strategy)].copy()
            for cutoff in CUTOFFS:
                values = evaluate_one_query(ranking, gold_for_question, cutoff)
                metric_rows.append({
                    'gold_view': gold_view_name,
                    'question_id': question_id,
                    'strategy': strategy,
                    'k': cutoff,
                    **values,
                })

per_query_metrics = pd.DataFrame(metric_rows)
if per_query_metrics.empty:
    raise RuntimeError('No se calcularon métricas. Verifica IDs de preguntas y retrieval_results.')
display(per_query_metrics.head())


,gold_view,question_id,strategy,k,evaluable_evidence,num_gold_evidence,num_gold_claims,evidence_hit_at_k,evidence_precision_at_k,evidence_recall_at_k,mrr_at_k,ap_at_k,claim_coverage_at_k,document_hit_at_k,page_hit_at_k,retrieved_relevant_evidence,retrieved_claims
0,human_only,Q002,semantic,1,True,3,5,1.0,1.000000,0.333333,1.0,0.333333,0.4,1.0,1.0,1,Q002-C03;Q002-C05
1,human_only,Q002,semantic,3,True,3,5,1.0,0.333333,0.333333,1.0,0.333333,0.4,1.0,1.0,1,Q002-C03;Q002-C05
2,human_only,Q002,semantic,5,True,3,5,1.0,0.200000,0.333333,1.0,0.333333,0.4,1.0,1.0,1,Q002-C03;Q002-C05
3,human_only,Q002,semantic,10,True,3,5,1.0,0.100000,0.333333,1.0,0.333333,0.4,1.0,1.0,1,Q002-C03;Q002-C05
4,human_only,Q002,lexical,1,True,3,5,0.0,0.000000,0.000000,0.0,0.000000,0.0,1.0,0.0,0,


In [15]:
per_query_metrics_path = FINAL_DIR / 'per_query_metrics_21082026_2027.csv'
per_query_metrics.to_csv(per_query_metrics_path, index=False)
print(f'Métricas por pregunta guardadas en: {per_query_metrics_path}')

Métricas por pregunta guardadas en: C:\Users\mamen\Documents\Python\RAGChatBot\evaluation_v2\run_20260809_170051\final_single_reviewer_v2\per_query_metrics_21082026_2027.csv


In [16]:
# Agregación macro: cada pregunta evaluable pesa lo mismo. MAP es la media de AP@k.
METRIC_COLUMNS = [
    'evidence_hit_at_k', 'evidence_precision_at_k', 'evidence_recall_at_k',
    'mrr_at_k', 'ap_at_k', 'claim_coverage_at_k',
    'document_hit_at_k', 'page_hit_at_k',
]

aggregate_metrics = (
    per_query_metrics
    .groupby(['gold_view', 'strategy', 'k'], as_index=False)
    .agg(
        n_questions=('question_id', 'nunique'),
        evidence_hit_at_k=('evidence_hit_at_k', 'mean'),
        evidence_precision_at_k=('evidence_precision_at_k', 'mean'),
        evidence_recall_at_k=('evidence_recall_at_k', 'mean'),
        mrr_at_k=('mrr_at_k', 'mean'),
        map_at_k=('ap_at_k', 'mean'),
        claim_coverage_at_k=('claim_coverage_at_k', 'mean'),
        document_hit_at_k=('document_hit_at_k', 'mean'),
        page_hit_at_k=('page_hit_at_k', 'mean'),
    )
)

# Los porcentajes facilitan interpretación, sin sustituir los valores decimales de las métricas.
display(aggregate_metrics.sort_values(['gold_view', 'k', 'strategy']).style.format({
    column: '{:.3f}' for column in aggregate_metrics.columns if column not in {'gold_view', 'strategy', 'k', 'n_questions'}
}))


,gold_view,strategy,k,n_questions,evidence_hit_at_k,evidence_precision_at_k,evidence_recall_at_k,mrr_at_k,map_at_k,claim_coverage_at_k,document_hit_at_k,page_hit_at_k
0,expanded,hybrid,1,31,0.419,0.419,0.221,0.419,0.221,0.289,0.484,0.419
4,expanded,lexical,1,31,0.129,0.129,0.039,0.129,0.039,0.069,0.387,0.129
8,expanded,semantic,1,31,0.323,0.323,0.174,0.323,0.174,0.228,0.613,0.355
1,expanded,hybrid,3,31,0.548,0.215,0.269,0.468,0.244,0.364,0.677,0.548
5,expanded,lexical,3,31,0.258,0.097,0.096,0.188,0.070,0.151,0.645,0.258
9,expanded,semantic,3,31,0.548,0.204,0.265,0.430,0.220,0.352,0.774,0.581
2,expanded,hybrid,5,31,0.581,0.142,0.303,0.474,0.251,0.397,0.839,0.613
6,expanded,lexical,5,31,0.484,0.110,0.254,0.241,0.108,0.321,0.774,0.484
10,expanded,semantic,5,31,0.548,0.135,0.278,0.430,0.227,0.355,0.839,0.581
3,expanded,hybrid,10,31,0.645,0.090,0.349,0.483,0.263,0.449,0.903,0.677


In [17]:
aggregate_metrics_path = FINAL_DIR / 'aggregate_metrics_21082026_2027.csv'
aggregate_metrics.to_csv(aggregate_metrics_path, index=False)
print(f'Métricas agregadas guardadas en: {aggregate_metrics_path}')

Métricas agregadas guardadas en: C:\Users\mamen\Documents\Python\RAGChatBot\evaluation_v2\run_20260809_170051\final_single_reviewer_v2\aggregate_metrics_21082026_2027.csv


# Conclusiones de esta versión

**Resultados pendientes de ejecución.**

### Candidatos procedentes de AND y OR

| Procedencia | Candidatos | Porcentaje |
|---|---:|---:|
| AND | 21 | 0,74% |
| OR | 2.829 | 99,26% |
| Total lexical | 2.850 | 100% |

- OR predomina las evidencias 

### ¿OR aumenta Evidence Recall y Claim Coverage?

Vista expanded top k = 10

| Métrica lexical | AND, v4 | AND+OR, v5 | Diferencia |
|---|---:|---:|---:|
| Evidence Hit@10 | 0,161 | 0,613 | +0,452 |
| Evidence Recall@10 | 0,093 | 0,342 | +0,249 |
| Claim Coverage@10 | 0,105 | 0,418 | +0,313 |
| Document Hit@10 | 0,194 | 0,903 | +0,710 |
| Page Hit@10 | 0,161 | 0,613 | +0,452 |


### ¿OR reduce Precision, MRR o MAP?

En lexical aislada: no
A k=10, vista expanded:

| Métrica | AND | AND+OR |
|---|---:|---:|
| Precision@10 | 0,019 | 0,090 |
| MRR@10 | 0,121 | 0,259 |
| MAP@10 | 0,053 | 0,126 |

Ninguna pregunta empeora en estas métricas. Esto es lógico: los candidatos AND permanecen delante y OR ocupa posiciones que anteriormente estaban vacías. Sin embargo, lexical sigue ordenando peor que semantic:

| Métrica @10 | Semantic | Lexical AND+OR |
|---|---:|---:|
| Precision | 0,094 | 0,090 |
| Recall | 0,377 | 0,342 |
| MRR | 0,447 | 0,259 |
| MAP | 0,244 | 0,126 |
| Claim Coverage | 0,488 | 0,418 |

Dentro de hybrid sí aparece degradación: 

Comparando el hybrid anterior con el actual, vista expanded, k=10:
| Métrica hybrid | v4 | v5 AND+OR | Diferencia |
|---|---:|---:|---:|
| Evidence Hit | 0,710 | 0,645 | −0,065 |
| Precision | 0,103 | 0,090 | −0,013 |
| Recall | 0,420 | 0,349 | −0,070 |
| MRR | 0,525 | 0,483 | −0,042 |
| MAP | 0,310 | 0,263 | −0,047 |
| Claim Coverage | 0,532 | 0,449 | −0,083 |

### Semantic frente a hybrid
Vista expanded, k=10:

| Métrica | Semantic | Hybrid |
|---|---:|---:|
| Evidence Hit | 0,677 | 0,645 |
| Precision | 0,094 | 0,090 |
| Recall | 0,377 | 0,349 |
| MRR | 0,447 | 0,483 |
| MAP | 0,244 | 0,263 |
| Claim Coverage | 0,488 | 0,449 |
| Document Hit | 0,903 | 0,903 |
| Page Hit | 0,710 | 0,677 |

Hybrid mejora MRR y MAP, pero pierde Hit, Recall y Claim Coverage.
Esto significa que hybrid tiene dos efectos opuestos:
- en algunas preguntas adelanta mucho la primera evidencia relevante;
- en otras desplaza completamente la evidencia fuera del top‑10.

### Comportamiento según K

En expanded:
- A k=1, hybrid es claramente mejor en Hit: 0,419 frente a 0,323 semantic.
- A k=3, ambos tienen el mismo Hit, pero hybrid tiene mejor MRR y MAP.
- A k=5, hybrid supera ligeramente a semantic en Hit, Recall, MRR, MAP y cobertura.
- A k=10, semantic recupera más evidencias y claims, aunque hybrid conserva mejor MRR y MAP.
Esto indica que hybrid adelanta algunas evidencias relevantes, pero también desplaza otras. Funciona mejor como optimizador de los primeros puestos que como recuperador de cobertura amplia.

### Otras conclusiones
- Los sistemas suelen llegar al documento correcto, pero no siempre a la página o evidencia exacta ya que en k=10 los tres documentos tienen el mismo Hit@Document


# Próximos pasos
1. Probar de nuevo pero solamente utilizando 20-30 candidatos OR
2. Aplicar pesos en el RFF a los resultados semánticos y léxicos, e incluso a los candidatos que provienen del AND y del OR.

# Próximos pasos tras observar resultados (21:15)

## Matriz de experimentos RRF

Esta sección compara siete configuraciones de hybrid. El baseline con OR máximo 100 y pesos 1/1/1 reutiliza los resultados ya calculados en las celdas anteriores y **no vuelve a ejecutar retrieval**.

Las otras seis configuraciones solo se ejecutan cuando `RUN_RRF_EXPERIMENTS = True`. Cada ejecución utiliza el mismo evidence set, profundidad, `RRF_K`, detector de idioma y funciones de métricas.

| Experimento | OR máximo | Semantic | AND | OR |
|---|---:|---:|---:|---:|
| Baseline OR=100 | 100 | 1,0 | 1,0 | 1,0 |
| Baseline OR=50 | 50 | 1,0 | 1,0 | 1,0 |
| Baseline OR=30 | 30 | 1,0 | 1,0 | 1,0 |
| OR moderado | 30 | 1,0 | 1,0 | 0,5 |
| Recomendado inicial | 30 | 1,0 | 1,0 | 0,2 |
| OR muy conservador | 20 | 1,0 | 1,0 | 0,1 |
| Ablación sin OR en hybrid | 30 | 1,0 | 1,0 | 0,0 |

Los pesos solo afectan a RRF. El límite OR controla cuántos candidatos relajados puede aportar la rama lexical antes de la fusión.

In [26]:
from datetime import datetime, timezone
import importlib
import app.retrieval_revised_v5 as retrieval_v5_module

# Recarga v5 para que un kernel que ya ejecutó la versión anterior utilice
# la nueva firma con pesos separados para semantic, AND y OR.
retrieval_v5_module = importlib.reload(retrieval_v5_module)
hybrid_search = retrieval_v5_module.hybrid_search
resolve_lexical_language = (
    retrieval_v5_module.resolve_lexical_language
)


# Mantener False permite revisar toda la configuración sin consultar la base
# de datos ni volver a generar embeddings. Cambiar a True ejecuta únicamente
# las seis configuraciones nuevas; el baseline OR=100 se reutiliza.
RUN_RRF_EXPERIMENTS = True

RRF_EXPERIMENTS = [
    {
        "experiment_order": 0,
        "experiment_id": "baseline_or100_w1_1_1",
        "experiment_name": "Baseline OR=100",
        "or_fallback_k": 100,
        "semantic_weight": 1.0,
        "lexical_and_weight": 1.0,
        "lexical_or_weight": 1.0,
        "reuse_existing_baseline": True,
    },
    {
        "experiment_order": 1,
        "experiment_id": "baseline_or50_w1_1_1",
        "experiment_name": "Baseline OR=50",
        "or_fallback_k": 50,
        "semantic_weight": 1.0,
        "lexical_and_weight": 1.0,
        "lexical_or_weight": 1.0,
        "reuse_existing_baseline": False,
    },
    {
        "experiment_order": 2,
        "experiment_id": "baseline_or30_w1_1_1",
        "experiment_name": "Baseline OR=30",
        "or_fallback_k": 30,
        "semantic_weight": 1.0,
        "lexical_and_weight": 1.0,
        "lexical_or_weight": 1.0,
        "reuse_existing_baseline": False,
    },
    {
        "experiment_order": 3,
        "experiment_id": "or30_moderate_w1_1_05",
        "experiment_name": "OR moderado",
        "or_fallback_k": 30,
        "semantic_weight": 1.0,
        "lexical_and_weight": 1.0,
        "lexical_or_weight": 0.5,
        "reuse_existing_baseline": False,
    },
    {
        "experiment_order": 4,
        "experiment_id": "or30_recommended_w1_1_02",
        "experiment_name": "Recomendado inicial",
        "or_fallback_k": 30,
        "semantic_weight": 1.0,
        "lexical_and_weight": 1.0,
        "lexical_or_weight": 0.2,
        "reuse_existing_baseline": False,
    },
    {
        "experiment_order": 5,
        "experiment_id": "or20_conservative_w1_1_01",
        "experiment_name": "OR muy conservador",
        "or_fallback_k": 20,
        "semantic_weight": 1.0,
        "lexical_and_weight": 1.0,
        "lexical_or_weight": 0.1,
        "reuse_existing_baseline": False,
    },
    {
        "experiment_order": 6,
        "experiment_id": "or30_ablation_w1_1_0",
        "experiment_name": "Ablación sin OR en hybrid",
        "or_fallback_k": 30,
        "semantic_weight": 1.0,
        "lexical_and_weight": 1.0,
        "lexical_or_weight": 0.0,
        "reuse_existing_baseline": False,
    },
]

rrf_experiment_design = pd.DataFrame(RRF_EXPERIMENTS)
display(rrf_experiment_design)

,experiment_order,experiment_id,experiment_name,or_fallback_k,semantic_weight,lexical_and_weight,lexical_or_weight,reuse_existing_baseline
0,0,baseline_or100_w1_1_1,Baseline OR=100,100,1.0,1.0,1.0,True
1,1,baseline_or50_w1_1_1,Baseline OR=50,50,1.0,1.0,1.0,False
2,2,baseline_or30_w1_1_1,Baseline OR=30,30,1.0,1.0,1.0,False
3,3,or30_moderate_w1_1_05,OR moderado,30,1.0,1.0,0.5,False
4,4,or30_recommended_w1_1_02,Recomendado inicial,30,1.0,1.0,0.2,False
5,5,or20_conservative_w1_1_01,OR muy conservador,20,1.0,1.0,0.1,False
6,6,or30_ablation_w1_1_0,Ablación sin OR en hybrid,30,1.0,1.0,0.0,False


In [27]:
def execute_retrieval(
    questions: pd.DataFrame,
    retrieval_depth: int,
    rrf_k: int,
    topic_filter: str | None,
    experiment: dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Ejecuta una configuración hybrid y devuelve resultados auditables.

    La función no ejecuta semantic y lexical como estrategias independientes:
    hybrid_search ya recupera ambas ramas y aplica RRF. Esto evita consultas
    adicionales que no dependen de los pesos del experimento.

    Todas las operaciones de retrieval de v5 son SELECT. Esta función no guarda
    archivos; la persistencia se realiza después con nombres timestamped.
    """

    retrieval_rows = []
    language_diagnostic_rows = []

    for question_row in questions.itertuples(index=False):
        question_id = str(question_row.question_id)
        query = str(question_row.question)

        # Una única resolución lingüística se reutiliza en la rama lexical
        # interna de hybrid para que el experimento sea reproducible.
        language_resolution = resolve_lexical_language(query)

        retrieved_chunks = hybrid_search(
            query,
            top_k=retrieval_depth,
            topic=topic_filter,
            rrf_k=rrf_k,
            candidate_k=retrieval_depth,
            language_resolution=language_resolution,
            or_fallback_k=int(experiment["or_fallback_k"]),
            semantic_weight=float(experiment["semantic_weight"]),
            lexical_and_weight=float(experiment["lexical_and_weight"]),
            lexical_or_weight=float(experiment["lexical_or_weight"]),
        )

        # Los recuentos AND/OR son metadatos de consulta repetidos en cada
        # resultado. El primer resultado basta para construir el diagnóstico.
        first_item = asdict(retrieved_chunks[0]) if retrieved_chunks else {}

        language_diagnostic_rows.append({
            "experiment_id": experiment["experiment_id"],
            "question_id": question_id,
            "detector_name": language_resolution.detector_name,
            "detected_language": language_resolution.detected_language,
            "language_confidence": language_resolution.confidence,
            "language_confidence_margin": language_resolution.confidence_margin,
            "postgres_ts_config": language_resolution.postgres_config,
            "language_supported": language_resolution.supported,
            "fallback_to_simple": language_resolution.fallback_to_simple,
            "language_note": language_resolution.note,
            "lexical_and_candidates": first_item.get(
                "lexical_and_candidates", 0
            ),
            "lexical_or_candidates_added": first_item.get(
                "lexical_or_candidates_added", 0
            ),
            "lexical_or_fallback_used": first_item.get(
                "lexical_or_fallback_used", False
            ),
        })

        seen = set()
        final_rank = 0

        for original_rank, chunk in enumerate(retrieved_chunks, start=1):
            item = asdict(chunk)
            dedupe_key = (str(item["doc_id"]), str(item["chunk_id"]))
            if dedupe_key in seen:
                continue

            seen.add(dedupe_key)
            final_rank += 1

            retrieval_rows.append({
                "experiment_order": experiment["experiment_order"],
                "experiment_id": experiment["experiment_id"],
                "experiment_name": experiment["experiment_name"],
                "or_fallback_k": experiment["or_fallback_k"],
                "semantic_weight": experiment["semantic_weight"],
                "lexical_and_weight": experiment["lexical_and_weight"],
                "lexical_or_weight": experiment["lexical_or_weight"],
                "question_id": question_id,
                "question": query,
                "strategy": "hybrid",
                "rank": final_rank,
                "original_rank": original_rank,
                "doc_id": item["doc_id"],
                "chunk_id": item["chunk_id"],
                "page_num": item["page_num"],
                "chunk_lang": item["lang"],
                "raw_score": item.get("score"),
                "semantic_rank": item.get("semantic_rank"),
                "lexical_rank": item.get("lexical_rank"),
                "semantic_distance": item.get("semantic_distance"),
                "lexical_score": item.get("lexical_score"),
                "lexical_query_mode": item.get("lexical_query_mode"),
                "ts_config": item.get("ts_config"),
                "rrf_k": rrf_k,
                "candidate_depth": retrieval_depth,
                "detector_name": language_resolution.detector_name,
                "detected_language": language_resolution.detected_language,
                "language_confidence": language_resolution.confidence,
                "language_confidence_margin": (
                    language_resolution.confidence_margin
                ),
                "postgres_ts_config": language_resolution.postgres_config,
                "language_supported": language_resolution.supported,
                "fallback_to_simple": (
                    language_resolution.fallback_to_simple
                ),
                "language_note": language_resolution.note,
                "lexical_and_candidates": item.get(
                    "lexical_and_candidates"
                ),
                "lexical_or_candidates_added": item.get(
                    "lexical_or_candidates_added"
                ),
                "lexical_or_fallback_used": item.get(
                    "lexical_or_fallback_used"
                ),
            })

    retrieval_frame = pd.DataFrame(retrieval_rows)
    if retrieval_frame.empty:
        raise RuntimeError(
            "Hybrid no devolvió resultados para el experimento "
            f"{experiment['experiment_id']}."
        )

    return retrieval_frame, pd.DataFrame(language_diagnostic_rows)

In [28]:
def calculate_experiment_metrics(
    retrieval_frame: pd.DataFrame,
    experiment: dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Reutiliza evaluate_one_query y las dos vistas gold del notebook."""

    metric_rows = []

    for gold_view_name, gold_view in gold_views.items():
        for question_row in questions.itertuples(index=False):
            question_id = str(question_row.question_id)
            gold_for_question = gold_view["by_question"].get(question_id)

            # Se conserva el mismo criterio que en la evaluación principal:
            # no se inventan negativos para preguntas sin gold evaluable.
            if gold_for_question is None:
                continue

            ranking = retrieval_frame.loc[
                retrieval_frame["question_id"].eq(question_id)
            ].copy()

            for cutoff in CUTOFFS:
                values = evaluate_one_query(
                    ranking,
                    gold_for_question,
                    cutoff,
                )
                metric_rows.append({
                    "experiment_order": experiment["experiment_order"],
                    "experiment_id": experiment["experiment_id"],
                    "experiment_name": experiment["experiment_name"],
                    "or_fallback_k": experiment["or_fallback_k"],
                    "semantic_weight": experiment["semantic_weight"],
                    "lexical_and_weight": experiment[
                        "lexical_and_weight"
                    ],
                    "lexical_or_weight": experiment[
                        "lexical_or_weight"
                    ],
                    "gold_view": gold_view_name,
                    "question_id": question_id,
                    "strategy": "hybrid",
                    "k": cutoff,
                    **values,
                })

    per_query_frame = pd.DataFrame(metric_rows)
    if per_query_frame.empty:
        raise RuntimeError(
            "No se calcularon métricas para "
            f"{experiment['experiment_id']}."
        )

    aggregate_frame = (
        per_query_frame
        .groupby(
            [
                "experiment_order",
                "experiment_id",
                "experiment_name",
                "or_fallback_k",
                "semantic_weight",
                "lexical_and_weight",
                "lexical_or_weight",
                "gold_view",
                "strategy",
                "k",
            ],
            as_index=False,
        )
        .agg(
            n_questions=("question_id", "nunique"),
            evidence_hit_at_k=("evidence_hit_at_k", "mean"),
            evidence_precision_at_k=(
                "evidence_precision_at_k", "mean"
            ),
            evidence_recall_at_k=("evidence_recall_at_k", "mean"),
            mrr_at_k=("mrr_at_k", "mean"),
            map_at_k=("ap_at_k", "mean"),
            claim_coverage_at_k=("claim_coverage_at_k", "mean"),
            document_hit_at_k=("document_hit_at_k", "mean"),
            page_hit_at_k=("page_hit_at_k", "mean"),
        )
    )

    return per_query_frame, aggregate_frame


def add_experiment_metadata(
    frame: pd.DataFrame,
    experiment: dict,
) -> pd.DataFrame:
    """Añade al baseline existente la configuración que no se reejecuta."""

    enriched = frame.copy()
    for column in [
        "experiment_order",
        "experiment_id",
        "experiment_name",
        "or_fallback_k",
        "semantic_weight",
        "lexical_and_weight",
        "lexical_or_weight",
    ]:
        enriched[column] = experiment[column]
    return enriched

In [29]:
# El baseline OR=100 ya fue calculado en las celdas superiores.
# Se reutilizan únicamente sus filas hybrid y no se ejecuta de nuevo.
# Las funciones, el gold y la configuración sí deben haberse cargado, pero los
# resultados OR=100 pueden recuperarse de los CSV ya existentes. Así se evita
# ejecutar de nuevo el baseline al abrir el notebook en un kernel nuevo.
required_support_objects = [
    "pd",
    "questions",
    "gold_views",
    "evaluate_one_query",
    "FINAL_DIR",
    "CUTOFFS",
    "RETRIEVAL_DEPTH",
    "RRF_K",
    "TOPIC_FILTER",
]
missing_support_objects = [
    name for name in required_support_objects
    if name not in globals()
]
if missing_support_objects:
    raise RuntimeError(
        "Ejecuta las celdas de importación, configuración, carga del gold "
        "y definición de métricas, pero no necesitas repetir retrieval. "
        "Faltan: " + ", ".join(missing_support_objects)
    )

baseline_paths = {
    "retrieval": FINAL_DIR
    / "retrieval_results_21082026_2027.csv",
    "per_query": FINAL_DIR
    / "per_query_metrics_21082026_2027.csv",
    "aggregate": FINAL_DIR
    / "aggregate_metrics_21082026_2027.csv",
}

missing_baseline_files = [
    path for path in baseline_paths.values()
    if not path.exists()
]
if missing_baseline_files:
    raise FileNotFoundError(
        "No se puede reutilizar el baseline OR=100. Faltan: "
        + ", ".join(str(path) for path in missing_baseline_files)
    )

if "retrieval_results" not in globals():
    retrieval_results = pd.read_csv(
        baseline_paths["retrieval"]
    )
if "per_query_metrics" not in globals():
    per_query_metrics = pd.read_csv(
        baseline_paths["per_query"]
    )
if "aggregate_metrics" not in globals():
    aggregate_metrics = pd.read_csv(
        baseline_paths["aggregate"]
    )

baseline_experiment = next(
    item for item in RRF_EXPERIMENTS
    if item["reuse_existing_baseline"]
)

baseline_per_query = add_experiment_metadata(
    per_query_metrics.loc[
        per_query_metrics["strategy"].eq("hybrid")
    ],
    baseline_experiment,
)

baseline_aggregate = add_experiment_metadata(
    aggregate_metrics.loc[
        aggregate_metrics["strategy"].eq("hybrid")
    ],
    baseline_experiment,
)

all_per_query_frames = [baseline_per_query]
all_aggregate_frames = [baseline_aggregate]
experiment_retrieval_frames = {
    baseline_experiment["experiment_id"]: retrieval_results.loc[
        retrieval_results["strategy"].eq("hybrid")
    ].copy()
}
saved_files = [{
    "experiment_id": baseline_experiment["experiment_id"],
    "artifact_type": "retrieval_existing_baseline",
    "path": str(
        baseline_paths["retrieval"]
    ),
}]

# Un timestamp UTC común identifica todos los archivos de una misma ejecución.
experiment_run_timestamp_utc = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%fZ")

EXPERIMENT_OUTPUT_DIR = (
    FINAL_DIR / "rrf_weight_experiments"
)

if RUN_RRF_EXPERIMENTS:
    EXPERIMENT_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    for experiment in RRF_EXPERIMENTS:
        if experiment["reuse_existing_baseline"]:
            continue

        print(
            "Ejecutando:",
            experiment["experiment_name"],
            experiment["experiment_id"],
        )

        experiment_retrieval, experiment_language = (
            execute_retrieval(
                questions=questions,
                retrieval_depth=RETRIEVAL_DEPTH,
                rrf_k=RRF_K,
                topic_filter=TOPIC_FILTER,
                experiment=experiment,
            )
        )

        experiment_per_query, experiment_aggregate = (
            calculate_experiment_metrics(
                experiment_retrieval,
                experiment,
            )
        )

        experiment_retrieval_frames[
            experiment["experiment_id"]
        ] = experiment_retrieval
        all_per_query_frames.append(experiment_per_query)
        all_aggregate_frames.append(experiment_aggregate)

        file_prefix = (
            f"{experiment['experiment_id']}_"
            f"{experiment_run_timestamp_utc}"
        )
        output_paths = {
            "retrieval": EXPERIMENT_OUTPUT_DIR
            / f"retrieval_results_{file_prefix}.csv",
            "language_diagnostics": EXPERIMENT_OUTPUT_DIR
            / f"language_diagnostics_{file_prefix}.csv",
            "per_query_metrics": EXPERIMENT_OUTPUT_DIR
            / f"per_query_metrics_{file_prefix}.csv",
            "aggregate_metrics": EXPERIMENT_OUTPUT_DIR
            / f"aggregate_metrics_{file_prefix}.csv",
        }

        # Nunca se sobrescribe silenciosamente una ejecución anterior.
        existing_paths = [
            path for path in output_paths.values()
            if path.exists()
        ]
        if existing_paths:
            raise FileExistsError(
                "Ya existen archivos para esta ejecución: "
                + ", ".join(str(path) for path in existing_paths)
            )

        experiment_retrieval.to_csv(
            output_paths["retrieval"],
            index=False,
        )
        experiment_language.to_csv(
            output_paths["language_diagnostics"],
            index=False,
        )
        experiment_per_query.to_csv(
            output_paths["per_query_metrics"],
            index=False,
        )
        experiment_aggregate.to_csv(
            output_paths["aggregate_metrics"],
            index=False,
        )

        for artifact_type, path in output_paths.items():
            saved_files.append({
                "experiment_id": experiment["experiment_id"],
                "artifact_type": artifact_type,
                "path": str(path),
            })
else:
    print(
        "Experimentos no ejecutados. Cambia "
        "RUN_RRF_EXPERIMENTS a True para ejecutar las seis "
        "configuraciones nuevas. El baseline OR=100 no se repetirá."
    )

rrf_experiment_per_query_metrics = pd.concat(
    all_per_query_frames,
    ignore_index=True,
)

rrf_experiment_aggregate_metrics = pd.concat(
    all_aggregate_frames,
    ignore_index=True,
)

saved_files_manifest = pd.DataFrame(saved_files)

Ejecutando: Baseline OR=50 baseline_or50_w1_1_1
Ejecutando: Baseline OR=30 baseline_or30_w1_1_1
Ejecutando: OR moderado or30_moderate_w1_1_05
Ejecutando: Recomendado inicial or30_recommended_w1_1_02
Ejecutando: OR muy conservador or20_conservative_w1_1_01
Ejecutando: Ablación sin OR en hybrid or30_ablation_w1_1_0


In [30]:
# Tabla comparativa completa: una fila por experimento, vista gold y k.
comparison_columns = [
    "experiment_order",
    "experiment_id",
    "experiment_name",
    "or_fallback_k",
    "semantic_weight",
    "lexical_and_weight",
    "lexical_or_weight",
    "gold_view",
    "k",
    "n_questions",
    "evidence_hit_at_k",
    "evidence_precision_at_k",
    "evidence_recall_at_k",
    "mrr_at_k",
    "map_at_k",
    "claim_coverage_at_k",
    "document_hit_at_k",
    "page_hit_at_k",
]

rrf_experiment_comparison = (
    rrf_experiment_aggregate_metrics[comparison_columns]
    .sort_values(
        ["gold_view", "k", "experiment_order"]
    )
    .reset_index(drop=True)
)

# Una vista compacta a k=5 y k=10 facilita comparar la calidad de los primeros
# puestos y la cobertura cuando se amplía el contexto.
display(
    rrf_experiment_comparison.loc[
        rrf_experiment_comparison["k"].isin([5, 10])
    ].style.format({
        column: "{:.3f}"
        for column in [
            "evidence_hit_at_k",
            "evidence_precision_at_k",
            "evidence_recall_at_k",
            "mrr_at_k",
            "map_at_k",
            "claim_coverage_at_k",
            "document_hit_at_k",
            "page_hit_at_k",
        ]
    })
)

if RUN_RRF_EXPERIMENTS:
    combined_output_paths = {
        "comparison": EXPERIMENT_OUTPUT_DIR
        / (
            "rrf_experiment_comparison_"
            f"{experiment_run_timestamp_utc}.csv"
        ),
        "all_per_query_metrics": EXPERIMENT_OUTPUT_DIR
        / (
            "rrf_experiment_per_query_metrics_"
            f"{experiment_run_timestamp_utc}.csv"
        ),
    }

    existing_paths = [
        path for path in combined_output_paths.values()
        if path.exists()
    ]
    if existing_paths:
        raise FileExistsError(
            "No se sobrescribirán archivos existentes: "
            + ", ".join(str(path) for path in existing_paths)
        )

    rrf_experiment_comparison.to_csv(
        combined_output_paths["comparison"],
        index=False,
    )
    rrf_experiment_per_query_metrics.to_csv(
        combined_output_paths["all_per_query_metrics"],
        index=False,
    )

    for artifact_type, path in combined_output_paths.items():
        saved_files_manifest.loc[
            len(saved_files_manifest)
        ] = {
            "experiment_id": "combined",
            "artifact_type": artifact_type,
            "path": str(path),
        }

display(saved_files_manifest)

,experiment_order,experiment_id,experiment_name,or_fallback_k,semantic_weight,lexical_and_weight,lexical_or_weight,gold_view,k,n_questions,evidence_hit_at_k,evidence_precision_at_k,evidence_recall_at_k,mrr_at_k,map_at_k,claim_coverage_at_k,document_hit_at_k,page_hit_at_k
14,0,baseline_or100_w1_1_1,Baseline OR=100,100,1.000000,1.000000,1.000000,expanded,5,31,0.581,0.142,0.303,0.474,0.251,0.397,0.839,0.613
15,1,baseline_or50_w1_1_1,Baseline OR=50,50,1.000000,1.000000,1.000000,expanded,5,31,0.581,0.142,0.278,0.476,0.247,0.384,0.774,0.581
16,2,baseline_or30_w1_1_1,Baseline OR=30,30,1.000000,1.000000,1.000000,expanded,5,31,0.548,0.135,0.272,0.468,0.245,0.367,0.742,0.548
17,3,or30_moderate_w1_1_05,OR moderado,30,1.000000,1.000000,0.500000,expanded,5,31,0.645,0.161,0.385,0.534,0.288,0.478,0.839,0.645
18,4,or30_recommended_w1_1_02,Recomendado inicial,30,1.000000,1.000000,0.200000,expanded,5,31,0.613,0.168,0.385,0.527,0.292,0.466,0.935,0.645
19,5,or20_conservative_w1_1_01,OR muy conservador,20,1.000000,1.000000,0.100000,expanded,5,31,0.645,0.174,0.405,0.543,0.311,0.482,0.935,0.677
20,6,or30_ablation_w1_1_0,Ablación sin OR en hybrid,30,1.000000,1.000000,0.000000,expanded,5,31,0.613,0.161,0.353,0.511,0.295,0.431,0.935,0.645
21,0,baseline_or100_w1_1_1,Baseline OR=100,100,1.000000,1.000000,1.000000,expanded,10,31,0.645,0.090,0.349,0.483,0.263,0.449,0.903,0.677
22,1,baseline_or50_w1_1_1,Baseline OR=50,50,1.000000,1.000000,1.000000,expanded,10,31,0.645,0.090,0.364,0.483,0.264,0.457,0.935,0.677
23,2,baseline_or30_w1_1_1,Baseline OR=30,30,1.000000,1.000000,1.000000,expanded,10,31,0.677,0.097,0.416,0.485,0.271,0.507,0.935,0.710


,experiment_id,artifact_type,path
0,baseline_or100_w1_1_1,retrieval_existing_baseline,C:\Users\mamen\Documents\Python\RAGChatBot\eva...
1,baseline_or50_w1_1_1,retrieval,C:\Users\mamen\Documents\Python\RAGChatBot\eva...
2,baseline_or50_w1_1_1,language_diagnostics,C:\Users\mamen\Documents\Python\RAGChatBot\eva...
3,baseline_or50_w1_1_1,per_query_metrics,C:\Users\mamen\Documents\Python\RAGChatBot\eva...
4,baseline_or50_w1_1_1,aggregate_metrics,C:\Users\mamen\Documents\Python\RAGChatBot\eva...
5,baseline_or30_w1_1_1,retrieval,C:\Users\mamen\Documents\Python\RAGChatBot\eva...
6,baseline_or30_w1_1_1,language_diagnostics,C:\Users\mamen\Documents\Python\RAGChatBot\eva...
7,baseline_or30_w1_1_1,per_query_metrics,C:\Users\mamen\Documents\Python\RAGChatBot\eva...
8,baseline_or30_w1_1_1,aggregate_metrics,C:\Users\mamen\Documents\Python\RAGChatBot\eva...
9,or30_moderate_w1_1_05,retrieval,C:\Users\mamen\Documents\Python\RAGChatBot\eva...


## Cómo ejecutar e interpretar la comparación

1. Ejecuta las celdas superiores de importación, configuración, carga del gold y definición de métricas. Puedes omitir la celda que vuelve a ejecutar retrieval.
2. Revisa la tabla `rrf_experiment_design`.
3. Cambia `RUN_RRF_EXPERIMENTS = True`.
4. Ejecuta en orden las celdas de esta sección. El baseline OR=100 se cargará desde sus CSV si no está en memoria.
5. Comprueba que aparecen las siete configuraciones en `rrf_experiment_comparison`.
6. Interpreta primero `human_only` y utiliza `expanded` como análisis de sensibilidad.
7. No elijas una configuración solo por una métrica: revisa conjuntamente Evidence Hit, Precision, Recall, MRR, MAP y Claim Coverage.
8. Utiliza development para seleccionar OR máximo y pesos. Reserva test para una única evaluación final.

Los archivos nuevos se guardan dentro de `final_single_reviewer_v2/rrf_weight_experiments/` con timestamp UTC. El baseline OR=100 conserva su archivo original y solo se incorpora a la tabla comparativa.

# Conclusiones de la comparación

- The experiments show that the original hybrid configuration was allowing the OR fallback to dominate the ranking. Limiting and down-weighting OR improves retrieval, but the exact optimal weight is not yet statistically established.
- Down-weighting OR lets semantic results recover the upper positions while preserving useful lexical matches.
- Retaining OR with a reduced weight is useful as a fallback, but harmful when treated as equally trustworthy as semantic or AND results.
- The retriever usually locates the correct document, but frequently fails to place the exact supporting chunk or all required evidence in the top 10. The remaining bottleneck is mainly chunk/evidence ranking, not document selection.

Best retrieval observed results: 
| Evaluation | Best balanced configuration | Hit@10 | Recall@10 | MRR@10 | MAP@10 | Claim Coverage@10 |
|---|---|---:|---:|---:|---:|---:|
| `expanded` | OR=30, OR weight=0.2 | 0.742 | 0.442 | 0.543 | 0.304 | 0.548 |
| `human_only` | OR=30, OR weight=0.5 | 0.389 | 0.259 | 0.184 | 0.114 | 0.295 |